

EXPERIMENTAL DESIGN CONFIGURATION GUIDE
========================================

1. INDEPENDENT VARIABLES (IVs)
    - Modify the 'IVs' dictionary to add or change factors you manipulate
    - Example: IVs["new_factor"] = [level1, level2, level3]
    - Use set_factor(IVs, "factor_name", [levels]) to add with validation

2. DEPENDENT VARIABLES (DVs)
    - Modify the 'DVs' dictionary to specify what you measure
    - Example: DVs["new_measure"] = ["continuous"] or ["categorical"]
    - Use set_factor(DVs, "measure_name", [type]) to add with validation

3. CONTROL VARIABLES (CVs)
    - Modify the 'CVs' dictionary to account for confounding factors
    - Example: CVs["new_cv"] = [level1, level2]
    - Use set_factor(CVs, "cv_name", [levels]) to add with validation

4. DESIGN SPECIFICATION (IV_designs)
    - For each IV, specify "within" (same participants see all levels) 
      or "between" (different participants see different levels)
    - Example: IV_designs["factor_name"] = "within"
    - Current settings:
      * xai_type: within (all participants see all XAI types)
      * tested_w_xai: between (different groups with/without XAI)

CURRENT CONFIGURATION:
- IVs: {xai_type, tested_w_xai}
- CVs: {age_group, gender}
- DVs: {accuracy, task_time}


In [8]:
from typing import Dict, List, Any

# Cell 0: define IVs, CVs, DVs

# iv_config: each factor maps to:
#   {'type': 'within'|'between', 'levels': [...], 'randomization': 'block'|'trial'}
# randomization is only used for within-subjects IVs.
iv_config: Dict[str, Dict] = {
    # Between-subjects: each participant is assigned to ONE XAI method group.
    "xai_method": {
        "type": "between",
        "levels": ["shap", "lime", "none"],
    },
    # Within-subjects: each participant sees both tested_w_xai levels.
    # Trial randomization means True/False is randomized across trials, not blocked.
    "tested_w_xai": {
        "type": "within",
        "randomization": "trial",
        "levels": [True, False],
    },
    # Uncomment to add more IVs:
    # "attribute_number": {"type": "between", "levels": [1, 2, 3, 4]},
    # "trial_number":     {"type": "within", "randomization": "trial", "levels": list(range(5, 11))},
}

CVs: Dict[str, List[Any]] = {
    "age_group": ["young", "adult", "senior"],
    "gender":    ["male", "female", "other"],
}

DVs: Dict[str, List[Any]] = {
    "accuracy":  ["continuous"],
    "task_time": ["continuous"],
}


def validate_iv_config(config: Dict[str, Dict]) -> None:
    for name, cfg in config.items():
        iv_type = cfg.get("type")
        if iv_type not in ("within", "between"):
            raise ValueError(f"IV '{name}': 'type' must be 'within' or 'between'")

        randomization = cfg.get("randomization", "block")
        if iv_type == "within" and randomization not in ("block", "trial"):
            raise ValueError(f"IV '{name}': 'randomization' must be 'block' or 'trial'")
        if iv_type == "between" and "randomization" in cfg:
            raise ValueError(f"IV '{name}': between-subjects IVs should not set 'randomization'")

        levels = cfg.get("levels", [])
        if not isinstance(levels, list) or len(levels) == 0:
            raise ValueError(f"IV '{name}': 'levels' must be a non-empty list.")
        if len(set(map(str, levels))) != len(levels):
            raise ValueError(f"IV '{name}': duplicate levels in {levels}")


def validate_factors(factors: Dict[str, List[Any]]) -> None:
    for name, levels in factors.items():
        if not isinstance(levels, list) or len(levels) == 0:
            raise ValueError(f"Factor '{name}' must map to a non-empty list.")
        if len(set(map(str, levels))) != len(levels):
            raise ValueError(f"Factor '{name}' has duplicate levels: {levels}")

validate_iv_config(iv_config)
validate_factors(CVs)
validate_factors(DVs)


def set_iv(
    name: str,
    iv_type: str,
    levels: List[Any],
    randomization: str = "block",
) -> None:
    cfg = {"type": iv_type, "levels": levels}
    if iv_type == "within":
        cfg["randomization"] = randomization
    validate_iv_config({name: cfg})
    iv_config[name] = cfg


def set_factor(factors: Dict[str, List[Any]], name: str, levels: List[Any]) -> None:
    validate_factors({name: levels})
    factors[name] = levels

print("iv_config:")
for name, cfg in iv_config.items():
    randomization = cfg.get("randomization", "-")
    print(
        f"  {name:<20} type={cfg['type']:<8}  "
        f"randomization={randomization:<5}  levels={cfg['levels']}"
    )
print(f"\nCVs: {list(CVs.keys())}")
print(f"DVs: {list(DVs.keys())}")

iv_config:
  xai_method           type=between   randomization=-      levels=['shap', 'lime', 'none']
  tested_w_xai         type=within    randomization=trial  levels=[True, False]

CVs: ['age_group', 'gender']
DVs: ['accuracy', 'task_time']


In [ ]:
# ── Dataset selection + stable train/test split ───────────────────────────────
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import os

dataset_ids = ["forest_cover", "wine_quality", "adult"]

def load_dataset_and_split(dataset_source, test_size=0.2, random_state=42):

    if isinstance(dataset_source, str) and dataset_source in dataset_ids:
        dataset_id = dataset_source

        if dataset_id == "wine_quality":
            df = pd.read_csv("../assets/original_datasets/wine_quality/winequality-red.csv")
            X_raw = df.drop(columns=["quality"]).values.astype(np.float32)
            y = (df["quality"] >= 7).astype(int).values
            feature_names = [c for c in df.columns if c != "quality"]
        elif dataset_id == "forest_cover":
            raise NotImplementedError("Add the CSV path and target column for 'forest_cover'.")
        elif dataset_id == "adult":
            raise NotImplementedError("Add the CSV path and target column for 'adult'.")
    else:
        csv_path = str(dataset_source)
        dataset_id = os.path.splitext(os.path.basename(csv_path))[0]
        df = pd.read_csv(csv_path)
        target_col = df.columns[-1]
        X_raw = df.drop(columns=[target_col]).values.astype(np.float32)
        y = df[target_col].values
        feature_names = [c for c in df.columns if c != target_col]

    raw_instance_ids = np.arange(len(df))
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X_raw)

    X_train, X_test, y_train, y_test, train_instance_ids, test_instance_ids = train_test_split(
        X_scaled,
        y,
        raw_instance_ids,
        test_size=test_size,
        random_state=random_state,
        stratify=y,
    )

    return (
        dataset_id, df, X_raw, y, feature_names,
        raw_instance_ids, scaler, X_scaled,
        X_train, X_test, y_train, y_test,
        train_instance_ids, test_instance_ids,
    )
selected_dataset_ids = ["wine_quality"]  # this workflow currently uses the first selected dataset

# Validate all selected ids
invalid_ids = [id for id in selected_dataset_ids if id not in dataset_ids]
if invalid_ids:
    raise ValueError(f"Invalid dataset ids: {invalid_ids}. Choose from {dataset_ids}")

if len(selected_dataset_ids) != 1:
    raise ValueError("This notebook currently expects exactly one selected dataset.")

dataset_id = selected_dataset_ids[0]

if dataset_id == "wine_quality":
    df = pd.read_csv("../assets/original_datasets/wine_quality/winequality-red.csv")
    X_raw = df.drop(columns=["quality"]).values.astype(np.float32)
    y = (df["quality"] >= 7).astype(int).values
    feature_names = [c for c in df.columns if c != "quality"]
else:
    raise NotImplementedError(
        f"Dataset split setup is not implemented for {dataset_id!r} in this notebook yet."
    )

# Stable raw-row ids. These are the ids written to explanation CSV instanceId.
raw_instance_ids = np.arange(len(df))

# Normalize features once, then preserve row ids during train/test split.
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_raw)

X_train, X_test, y_train, y_test, train_instance_ids, test_instance_ids = train_test_split(
    X_scaled,
    y,
    raw_instance_ids,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Dataset   : {dataset_id}  ({X_scaled.shape[0]} rows, {X_scaled.shape[1]} features)")
print(f"Train set : {X_train.shape[0]} samples  ({X_train.shape[0]/len(y)*100:.0f}%)")
print(f"Test set  : {X_test.shape[0]}  samples  ({X_test.shape[0]/len(y)*100:.0f}%)")
print(f"Class balance (train) -> class 0: {(y_train==0).sum()}  class 1: {(y_train==1).sum()}")
print(f"First test instanceIds: {test_instance_ids[:10].tolist()}")

Dataset   : wine_quality  (1599 rows, 11 features)
Train set : 1279 samples  (80%)
Test set  : 320  samples  (20%)
Class balance (train) -> class 0: 1105  class 1: 174
First test instanceIds: [434, 213, 1447, 231, 763, 1472, 472, 1327, 766, 19]


In [10]:
# ── Experimental Design: Counterbalancing & Trial Arrangement ─────────────────
# Uses iv_config, CVs defined in Cell 0 and selected_dataset_ids above.
import sys, os, csv
sys.path.insert(0, os.path.abspath(".."))

import importlib
import src.experiment_design.counterbalance as counterbalance
importlib.reload(counterbalance)

from src.experiment_design.counterbalance import (
    factorial_conditions,
    split_ivs_by_design_role,
    make_within_condition_order_labels,
    choose_counterbalancing,
    assign_participants,
    build_trial_sequence,
    export_trials_csv,
    export_trials_json,
    export_design_summary,
)

# ── 1. Split iv_config into between / block-within / trial-within ─────────────
between_ivs, block_within_ivs, trial_within_ivs = split_ivs_by_design_role(iv_config)
within_ivs = {**block_within_ivs, **trial_within_ivs}

print("IV split:")
print(f"  between-subjects       : {between_ivs}")
print(f"  block-counterbalanced  : {block_within_ivs}")
print(f"  trial-randomized       : {trial_within_ivs}")

# ── 2. All factorial conditions ────────────────────────────────────────────────
all_conditions = factorial_conditions({k: v["levels"] for k, v in iv_config.items()})
print(f"\nAll factorial conditions ({len(all_conditions)} total):")
for c in all_conditions:
    print(f"  {c}")

# ── 3. Counterbalancing strategy for BLOCK-level within-subjects IVs ──────────
# Trial-randomized within IVs are NOT counterbalanced as blocks.
within_labels = make_within_condition_order_labels(block_within_ivs)
orders, strategy = choose_counterbalancing(within_labels)

print(f"\nCounterbalancing strategy : {strategy}")
print(f"Number of orders          : {len(orders)}")
print("\nOrders (each row = one participant's block sequence):")
for i, order in enumerate(orders):
    print(f"  Order {i+1:>2}: {order}")

# ── 4. Participant assignment ──────────────────────────────────────────────────
# Use a fixed participant count per between-subjects condition.
# In the final study, set this value from power analysis.
PARTICIPANTS_PER_BETWEEN_CONDITION = 25

between_groups = factorial_conditions(between_ivs) if between_ivs else [{}]
n_between_groups = len(between_groups)
n_participants = PARTICIPANTS_PER_BETWEEN_CONDITION * n_between_groups

print(f"\nParticipants per between-subjects condition: {PARTICIPANTS_PER_BETWEEN_CONDITION}")
print(f"Number of between-subjects conditions     : {n_between_groups}")
print(f"Total n_participants                      : {n_participants}")

assignments = assign_participants(n_participants, orders, between_ivs or None)
print(f"\nParticipant assignments ({len(assignments)} total):")
for a in assignments:
    between_cols = {k: v for k, v in a.items() if k not in ("participantId", "within_order")}
    print(f"  P{a['participantId']:>02d}  {between_cols}  ->  {a['within_order']}")

# ── 5. Load instance pool from explanation CSV ─────────────────────────────────
dataset_id   = selected_dataset_ids[0]
explain_csv  = f"generated_explanation/de_mlp_{dataset_id}.csv"

def _load_csv(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

instance_pool = _load_csv(explain_csv)
print(f"\nInstance pool: {len(instance_pool)} instances from '{explain_csv}'")

# ── 6. CVs -> controlled_vars columns in every trial row ──────────────────────
# dataId already stores the dataset id, so do not duplicate it as dataset.
controlled_vars = {
    "model_type": "mlp",
    **{f"CV_{k}_levels": "|".join(str(v) for v in vals) for k, vals in CVs.items()},
}

# ── 7. Build flat trial sequence ───────────────────────────────────────────────
# Total number of trials per participant, not per condition.
# For balanced trial randomization, this must divide evenly across:
#   block-level within conditions x trial-randomized condition combos.
TRIALS_PER_PARTICIPANT = 10

trials = build_trial_sequence(
    assignments=assignments,
    instance_pool=instance_pool,
    trials_per_participant=TRIALS_PER_PARTICIPANT,
    controlled_vars=controlled_vars,
    id_map={"dataId": "dataId", "instanceId": "instanceId"},
    trial_randomized_ivs=trial_within_ivs or None,
    trial_randomization_strategy="balanced",
    instance_wise_explanation=False,  # set True later to include explanation columns
    shuffle_instances=True,
    seed=42,
)

print(f"\nTrial sequence:")
print(f"  {len(assignments)} participants x {TRIALS_PER_PARTICIPANT} trials each")
print(f"  Total trial rows : {len(trials)}")

# ── 8. Export ──────────────────────────────────────────────────────────────────
out_dir = "experiment_output"
os.makedirs(out_dir, exist_ok=True)

csv_path     = export_trials_csv(trials,  f"{out_dir}/trials.csv")
json_path    = export_trials_json(trials, f"{out_dir}/trials.json")
summary_path = export_design_summary(
    iv_config={k: v["levels"] for k, v in iv_config.items()},
    between_ivs=between_ivs,
    within_ivs=within_ivs,
    block_within_ivs=block_within_ivs,
    trial_within_ivs=trial_within_ivs,
    trial_randomization_strategy="balanced",
    trials_per_participant=TRIALS_PER_PARTICIPANT,
    strategy=strategy,
    orders=orders,
    assignments=assignments,
    path=f"{out_dir}/design_summary.json",
)
print(f"\nExported:")
print(f"  CSV     : {csv_path}")
print(f"  JSON    : {json_path}")
print(f"  Summary : {summary_path}")

# ── 9. Sample trial rows ───────────────────────────────────────────────────────
print("\nSample trial rows (first 10):")
key_cols = [
    "participantId", "trialId", "block", "trialWithinBlock", "withinCondition",
    *between_ivs.keys(), *block_within_ivs.keys(), *trial_within_ivs.keys(),
    "dataId", "instanceId",
]
for t in trials:
    print({k: t[k] for k in key_cols if k in t})

IV split:
  between-subjects       : {'xai_method': ['shap', 'lime', 'none']}
  block-counterbalanced  : {}
  trial-randomized       : {'tested_w_xai': [True, False]}

All factorial conditions (6 total):
  {'xai_method': 'shap', 'tested_w_xai': True}
  {'xai_method': 'shap', 'tested_w_xai': False}
  {'xai_method': 'lime', 'tested_w_xai': True}
  {'xai_method': 'lime', 'tested_w_xai': False}
  {'xai_method': 'none', 'tested_w_xai': True}
  {'xai_method': 'none', 'tested_w_xai': False}

Counterbalancing strategy : complete_counterbalancing
Number of orders          : 1

Orders (each row = one participant's block sequence):
  Order  1: [{'withinCondition': 'single_condition'}]

Participants per between-subjects condition: 25
Number of between-subjects conditions     : 3
Total n_participants                      : 75

Participant assignments (75 total):
  P01  {'xai_method': 'shap'}  ->  [{'withinCondition': 'single_condition'}]
  P02  {'xai_method': 'lime'}  ->  [{'withinCondition': 'sing

# Convert json files exported from UI to trial info

In [11]:
import json
import csv
import os

from src.experiment_design.counterbalance import (
    factorial_conditions,
    split_ivs_by_design_role,
    make_within_condition_order_labels,
    choose_counterbalancing,
    assign_participants,
    build_trial_sequence,
    export_trials_csv,
    export_trials_json,
    export_design_summary,
)

with open("/Users/wangzhuoyulucas/Documents/GitHub/xaikit-test-api/src/experiment_design/example_json_from_ui/template.json", "r", encoding="utf-8") as f:
    config = json.load(f)

iv_config = config["ivs"]
CVs = config["cvs"]
DVs = config["dvs"]

dataset_cfg = config["dataset"]
sampling_cfg = config["sampling"]
output_cfg = config["output"]

between_ivs, block_within_ivs, trial_within_ivs = split_ivs_by_design_role(iv_config)
within_ivs = {**block_within_ivs, **trial_within_ivs}

within_labels = make_within_condition_order_labels(block_within_ivs)
orders, strategy = choose_counterbalancing(within_labels)

between_groups = factorial_conditions(between_ivs) if between_ivs else [{}]
n_participants = (
    sampling_cfg["participants_per_between_condition"]
    * len(between_groups)
)

assignments = assign_participants(n_participants, orders, between_ivs or None)

with open(dataset_cfg["explanation_csv"], newline="", encoding="utf-8") as f:
    instance_pool = list(csv.DictReader(f))

controlled_vars = {
    "model_type": dataset_cfg["model_type"],
    **{
        f"CV_{k}_levels": "|".join(str(v) for v in vals)
        for k, vals in CVs.items()
    },
}

trials = build_trial_sequence(
    assignments=assignments,
    instance_pool=instance_pool,
    trials_per_participant=sampling_cfg["trials_per_participant"],
    controlled_vars=controlled_vars,
    id_map=dataset_cfg["id_map"],
    trial_randomized_ivs=trial_within_ivs or None,
    trial_randomization_strategy=sampling_cfg["trial_randomization_strategy"],
    instance_wise_explanation=sampling_cfg.get("instance_wise_explanation", False),
    shuffle_instances=sampling_cfg["shuffle_instances"],
    seed=config["seed"],
)

out_dir = output_cfg["out_dir"]
os.makedirs(out_dir, exist_ok=True)

csv_path = export_trials_csv(trials, f"{out_dir}/{output_cfg['trials_csv']}")
json_path = export_trials_json(trials, f"{out_dir}/{output_cfg['trials_json']}")
summary_path = export_design_summary(
    iv_config={k: v["levels"] for k, v in iv_config.items()},
    between_ivs=between_ivs,
    within_ivs=within_ivs,
    block_within_ivs=block_within_ivs,
    trial_within_ivs=trial_within_ivs,
    trial_randomization_strategy=sampling_cfg["trial_randomization_strategy"],
    trials_per_participant=sampling_cfg["trials_per_participant"],
    strategy=strategy,
    orders=orders,
    assignments=assignments,
    path=f"{out_dir}/{output_cfg['summary_json']}",
)

In [12]:
# ── Train default MLP on selected dataset split ────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from src.ai_models import ModelManager

# The dataset cell above prepares:
#   dataset_id, X_train, X_test, y_train, y_test, train_instance_ids, test_instance_ids

manager = ModelManager()
model = manager.create_model(
    dataset=dataset_id,
    model_type="mlp",           # default PyTorch MLP
    input_dim=X_train.shape[1],
    num_classes=2,
)
print(f"\nModel info: {model.get_info()}")

print("\nTraining MLP ...")
training_info = manager.train(
    X_train, y_train,
    epochs=300,
    batch_size=1000,
)
print(f"Training done: {training_info}")

test_accuracy = manager.evaluate(X_test, y_test)
print(f"\nTest accuracy : {test_accuracy:.4f}  ({test_accuracy*100:.1f}%)")

# Expose references for downstream cells (XAI adapter)
trained_model  = model          # UnifiedModel wrapper
trained_engine = model.engine   # raw MLPEngine (needed by xai_adapter)
train_data_X   = X_train
test_data_X    = X_test
test_data_y    = y_test

✓ Created new mlp (coxam) for 'wine_quality'

Model info: {'dataset': 'wine_quality', 'model_type': 'mlp', 'framework': 'pytorch', 'input_dim': 11, 'num_classes': 2, 'cognitive_agent': 'coxam', 'is_trained': False}

Training MLP ...
Epoch 1  loss=0.7456
Epoch 2  loss=0.6954
Epoch 3  loss=0.6490
Epoch 4  loss=0.6072
Epoch 5  loss=0.5637
Epoch 6  loss=0.5398
Epoch 7  loss=0.5072
Epoch 8  loss=0.4865
Epoch 9  loss=0.4723
Epoch 10  loss=0.4640
Epoch 11  loss=0.4403
Epoch 12  loss=0.4181
Epoch 13  loss=0.4086
Epoch 14  loss=0.4010
Epoch 15  loss=0.4059
Epoch 16  loss=0.4152
Epoch 17  loss=0.4110
Epoch 18  loss=0.3746
Epoch 19  loss=0.4037
Epoch 20  loss=0.3952
Epoch 21  loss=0.3718
Epoch 22  loss=0.3921
Epoch 23  loss=0.3864
Epoch 24  loss=0.3681
Epoch 25  loss=0.3981
Epoch 26  loss=0.4045
Epoch 27  loss=0.3892
Epoch 28  loss=0.3759
Epoch 29  loss=0.3614
Epoch 30  loss=0.3558
Epoch 31  loss=0.3796
Epoch 32  loss=0.3809
Epoch 33  loss=0.3970
Epoch 34  loss=0.3782
Epoch 35  loss=0.3883
Epoch 

In [13]:
# ── Generate explanation CSVs from xai_method / xai_type IV ───────────────────
# Uses objects exposed by the training cell:
#   trained_engine, train_data_X, test_data_X, y_train
import os
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd

from src.xai_adapter import create_xai_method_from_engine

DATASET_ID = dataset_id
MODEL_NAME = "mlp"
OUTPUT_DIR = Path("generated_explanation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Support either IV name. Your current notebook uses xai_method.
xai_iv_name = "xai_type" if "xai_type" in iv_config else "xai_method"
xai_methods = [m for m in iv_config[xai_iv_name]["levels"]]

if "feature_names" not in globals():
    feature_names = [f"feature_{i}" for i in range(test_data_X.shape[1])]

train_labels_for_xai = y_train if "y_train" in globals() else None
train_data_for_xai = SimpleNamespace(
    X=train_data_X,
    y=train_labels_for_xai,
    feature_names=feature_names,
)

if "test_instance_ids" not in globals():
    raise ValueError("Run the dataset selection/split cell before generating explanations.")
instance_ids = np.asarray(test_instance_ids)
predictions = np.argmax(trained_engine.predict(test_data_X), axis=1)

# Optional method-specific settings. Keep these modest for notebook speed.
method_kwargs = {
    "shap": {"n_background_samples": 30},
    "shap_kernel": {"n_background_samples": 30},
    "lime": {"num_samples": 1000},
}

saved_paths = []
explanation_dfs = []

for method_name in xai_methods:
    method_key = str(method_name).lower()

    if method_key in {"none", "no_xai", "control"}:
        print(f"Skipping adapter generation for xai method: {method_name}")
        continue

    print(f"\nGenerating explanations for xai method: {method_name}")
    try:
        explainer = create_xai_method_from_engine(
            method_key,
            engine=trained_engine,
            train_data=train_data_for_xai,
            preprocessing_fn=lambda x: np.asarray(x, dtype=np.float32),
            target=1,
            **method_kwargs.get(method_key, {}),
        )
        result = explainer.explain(test_data_X)
        explanation_df = result.to_explanation_df(
            instance_ids=instance_ids,
            predictions=predictions,
            dataset_id=DATASET_ID,
            model_name=MODEL_NAME,
        )

        # Make the method label match the IV level, e.g. "shap" instead of
        # adapter-internal names like "shap_kernel".
        explanation_df["expMethod"] = method_key

        out_path = OUTPUT_DIR / f"{method_key}_{MODEL_NAME}_{DATASET_ID}.csv"
        explanation_df.to_csv(out_path, index=False)

        saved_paths.append(out_path)
        explanation_dfs.append(explanation_df)
        print(f"  Saved: {out_path}  shape={explanation_df.shape}")
    except Exception as exc:
        print(f"  Skipped {method_name}: {type(exc).__name__}: {exc}")

if explanation_dfs:
    combined_df = pd.concat(explanation_dfs, ignore_index=True)
    combined_path = OUTPUT_DIR / f"de_{MODEL_NAME}_{DATASET_ID}.csv"
    combined_df.to_csv(combined_path, index=False)
    print(f"\nCombined explanation CSV: {combined_path}  shape={combined_df.shape}")
else:
    combined_path = None
    print("\nNo explanation CSVs were generated. Check installed XAI dependencies.")

print("\nSaved files:")
for path in saved_paths:
    print(f"  {path}")


Generating explanations for xai method: shap


100%|██████████| 320/320 [00:05<00:00, 55.77it/s]


  Saved: generated_explanation/shap_mlp_wine_quality.csv  shape=(320, 18)

Generating explanations for xai method: lime
  Saved: generated_explanation/lime_mlp_wine_quality.csv  shape=(320, 18)
Skipping adapter generation for xai method: none

Combined explanation CSV: generated_explanation/de_mlp_wine_quality.csv  shape=(640, 18)

Saved files:
  generated_explanation/shap_mlp_wine_quality.csv
  generated_explanation/lime_mlp_wine_quality.csv


# cognitive models

In [14]:
# ── Draft cognitive model + experiment executor ──────────────────────────────
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence

import numpy as np
import pandas as pd


def load_explanation_pool(
    explanation_csv: str | Path = "generated_explanation/de_mlp_wine_quality.csv",
) -> pd.DataFrame:
    """Load generated explanations used by the trial generator."""
    return pd.read_csv(explanation_csv)


def get_trial_instance_attributes(
    trial_info: Dict[str, Any],
    raw_dataset: pd.DataFrame,
    *,
    label_column: str = "quality",
) -> Dict[str, float]:
    """
    Extract raw feature values for the trial's instanceId.

    Assumes instanceId is the original raw dataset row index created during the
    dataset split cell.
    """
    instance_id = int(trial_info["instanceId"])
    row = raw_dataset.iloc[instance_id]
    feature_row = row.drop(labels=[label_column], errors="ignore")
    return {k: float(v) for k, v in feature_row.items()}


def get_trial_ai_prediction(
    trial_info: Dict[str, Any],
    explanation_pool: pd.DataFrame,
) -> Optional[int]:
    """
    Return the AI model prediction for this instance.

    In the generated explanation CSV, `pred` is the trained AI model prediction
    passed into `result.to_explanation_df(...)`, not a separate XAI prediction.
    """
    if "pred" not in explanation_pool.columns:
        return None

    instance_id = int(trial_info["instanceId"])
    matches = explanation_pool[explanation_pool["instanceId"].astype(int) == instance_id]
    if matches.empty:
        return None
    return int(matches.iloc[0]["pred"])


def get_trial_instance_explanation(
    trial_info: Dict[str, Any],
    explanation_pool: pd.DataFrame,
) -> Dict[str, float]:
    """
    Extract explanation values for this trial using trial xai_method + instanceId.

    If xai_method is 'none' or the matching explanation row is absent, returns {}.
    """
    xai_method = str(trial_info.get("xai_method", trial_info.get("xai_type", "none"))).lower()
    if xai_method in {"none", "no_xai", "control"}:
        return {}

    instance_id = int(trial_info["instanceId"])
    matches = explanation_pool[
        (explanation_pool["instanceId"].astype(int) == instance_id)
        & (explanation_pool["expMethod"].astype(str).str.lower() == xai_method)
    ]
    if matches.empty:
        return {}

    row = matches.iloc[0]
    explanation_cols = [c for c in explanation_pool.columns if c.startswith("a") and c.endswith("_i")]
    explanation = {c: float(row[c]) for c in explanation_cols}

    for optional_col in ["pred", "i_max", "intercept"]:
        if optional_col in row:
            explanation[optional_col] = float(row[optional_col])

    return explanation


def build_single_trial_cognitive_input(
    trial_info: Dict[str, Any],
    raw_dataset: pd.DataFrame,
    explanation_pool: pd.DataFrame,
    *,
    label_column: str = "quality",
) -> Dict[str, Any]:
    """Combine trial metadata, raw instance attributes, AI pred, and optional explanation."""
    return {
        "trial_info": dict(trial_info),
        "instance_attributes": get_trial_instance_attributes(
            trial_info,
            raw_dataset,
            label_column=label_column,
        ),
        "instance_explanation": get_trial_instance_explanation(
            trial_info,
            explanation_pool,
        ),
        "ai_prediction": get_trial_ai_prediction(trial_info, explanation_pool),
    }


def dummy_cognitive_model(
    cognitive_params: Dict[str, float],
    dvs: Dict[str, List[Any]],
    trial_data: Dict[str, Any],
) -> Dict[str, Any]:
    """
    Draft placeholder cognitive model.

    Returns requested DVs plus:
      - agent_prediction: bool, True when accuracy-like probability > 0.5
      - ai_prediction: int | None, the AI model label from explanation CSV `pred`
    """
    trial_info = trial_data.get("trial_info", {})
    explanation = trial_data.get("instance_explanation", {}) or {}

    has_xai = bool(explanation) and str(trial_info.get("xai_method", "none")).lower() != "none"
    attr_values = [abs(v) for k, v in explanation.items() if k.startswith("a") and k.endswith("_i")]
    explanation_strength = float(np.mean(attr_values)) if attr_values else 0.0

    base_accuracy = float(cognitive_params.get("base_accuracy", 0.65))
    xai_accuracy_bonus = float(cognitive_params.get("xai_accuracy_bonus", 0.10))
    explanation_strength_weight = float(cognitive_params.get("explanation_strength_weight", 0.05))

    accuracy_probability = base_accuracy
    if has_xai:
        accuracy_probability += xai_accuracy_bonus
    accuracy_probability += explanation_strength_weight * explanation_strength
    accuracy_probability = float(np.clip(accuracy_probability, 0.0, 1.0))

    base_task_time = float(cognitive_params.get("base_task_time", 8.0))
    xai_time_cost = float(cognitive_params.get("xai_time_cost", 2.0))
    explanation_time_weight = float(cognitive_params.get("explanation_time_weight", 1.0))

    task_time = base_task_time
    if has_xai:
        task_time += xai_time_cost
    task_time += explanation_time_weight * explanation_strength
    task_time = float(max(task_time, 0.0))

    outputs: Dict[str, Any] = {}
    for dv_name in dvs:
        dv_key = dv_name.lower()
        if "accuracy" in dv_key or "prob" in dv_key or "correct" in dv_key:
            outputs[dv_name] = accuracy_probability
        elif "time" in dv_key or "rt" in dv_key or "duration" in dv_key:
            outputs[dv_name] = task_time
        else:
            outputs[dv_name] = accuracy_probability

    outputs["agent_prediction"] = bool(accuracy_probability > 0.5)
    outputs["ai_prediction"] = trial_data.get("ai_prediction")
    return outputs


def _trial_rows_for_mode(
    trials_df: pd.DataFrame,
    mode: str,
    *,
    participant_id: Optional[int] = None,
    condition_filter: Optional[Dict[str, Any]] = None,
) -> pd.DataFrame:
    """Select which trial rows the executor should run."""
    mode = mode.lower()
    selected = trials_df.copy()

    if mode in {"trial", "trial_by_trial"}:
        return selected.iloc[:1].copy()

    if mode in {"participant", "participant_by_participant"}:
        if participant_id is None:
            participant_id = int(selected["participantId"].iloc[0])
        return selected[selected["participantId"].astype(int) == int(participant_id)].copy()

    if mode in {"condition", "whole_condition"}:
        if not condition_filter:
            raise ValueError("condition_filter is required for mode='whole_condition'.")
        for col, value in condition_filter.items():
            selected = selected[selected[col].astype(str) == str(value)]
        return selected.copy()

    if mode in {"experiment", "whole_experiment"}:
        return selected.copy()

    raise ValueError(
        "mode must be one of: trial_by_trial, participant_by_participant, "
        "whole_condition, whole_experiment"
    )


def run_experiment_executor(
    trials: List[Dict[str, Any]] | pd.DataFrame,
    cognitive_params: Dict[str, float],
    dvs: Dict[str, List[Any]],
    raw_dataset: pd.DataFrame,
    explanation_pool: pd.DataFrame,
    *,
    mode: str = "whole_experiment",
    participant_id: Optional[int] = None,
    condition_filter: Optional[Dict[str, Any]] = None,
    cognitive_model=dummy_cognitive_model,
    label_column: str = "quality",
    explanation_prefix: str = "exp_",
    cognitive_param_prefix: str = "cog_param_",
) -> pd.DataFrame:
    """
    Run cognitive agents over selected trials and append explanations/results.

    Modes:
      - trial_by_trial: runs the first selected trial row
      - participant_by_participant: runs all trials for one participant
      - whole_condition: runs rows matching condition_filter, e.g. {'xai_method': 'shap'}
      - whole_experiment: runs all trial rows

    Appended columns include:
      - explanation columns used, prefixed by exp_, e.g. exp_a0_i, exp_pred
      - cognitive parameters used, prefixed by cog_param_
      - cognitive model outputs, e.g. accuracy, task_time, agent_prediction
      - cognitive_correct_vs_ai: agent_prediction == AI model pred from CSV
    """
    trials_df = pd.DataFrame(trials).copy()
    selected = _trial_rows_for_mode(
        trials_df,
        mode,
        participant_id=participant_id,
        condition_filter=condition_filter,
    )

    executed_rows = []
    for _, trial_row in selected.iterrows():
        trial_info = trial_row.to_dict()
        trial_context = build_single_trial_cognitive_input(
            trial_info,
            raw_dataset,
            explanation_pool,
            label_column=label_column,
        )
        model_outputs = cognitive_model(cognitive_params, dvs, trial_context)

        explanation = trial_context["instance_explanation"]
        explanation_cols = {
            f"{explanation_prefix}{k}": v
            for k, v in explanation.items()
        }
        cognitive_param_cols = {
            f"{cognitive_param_prefix}{k}": v
            for k, v in cognitive_params.items()
        }

        ai_prediction = model_outputs.get("ai_prediction")
        agent_prediction = model_outputs.get("agent_prediction")
        if ai_prediction is None:
            cognitive_correct_vs_ai = None
        else:
            cognitive_correct_vs_ai = bool(int(agent_prediction) == int(ai_prediction))

        executed_rows.append({
            **trial_info,
            **explanation_cols,
            **cognitive_param_cols,
            **model_outputs,
            "cognitive_correct_vs_ai": cognitive_correct_vs_ai,
        })

    return pd.DataFrame(executed_rows)


# Example usage after trials and explanations exist:
explanation_pool = load_explanation_pool("generated_explanation/de_mlp_wine_quality.csv")
executed = run_experiment_executor(
    trials,
    cognitive_params={"base_accuracy": 0.7},
    dvs=DVs,
    raw_dataset=df,
    explanation_pool=explanation_pool,
    mode="participant_by_participant",
    participant_id=1,
)
executed.head()

,participantId,trialId,block,trialWithinBlock,withinCondition,tested_w_xai,xai_method,dataId,instanceId,model_type,...,exp_a10_i,exp_pred,exp_i_max,exp_intercept,cog_param_base_accuracy,accuracy,task_time,agent_prediction,ai_prediction,cognitive_correct_vs_ai
0,1,1,1,1,single_condition,False,shap,wine_quality,1462,mlp,...,-1.000000,0.0,0.037048,0.136645,0.7,0.818564,10.371279,True,0,False
1,1,2,1,2,single_condition,True,shap,wine_quality,721,mlp,...,-1.000000,0.0,0.070114,0.136645,0.7,0.813830,10.276608,True,0,False
2,1,3,1,3,single_condition,True,shap,wine_quality,521,mlp,...,-1.000000,0.0,0.078522,0.136645,0.7,0.808297,10.165940,True,0,False
3,1,4,1,4,single_condition,False,shap,wine_quality,861,mlp,...,0.487355,0.0,0.057022,0.136645,0.7,0.818592,10.371838,True,0,False
4,1,5,1,5,single_condition,False,shap,wine_quality,344,mlp,...,-0.081259,0.0,0.058667,0.136645,0.7,0.818284,10.365672,True,0,False
